# GNR653 Lecture 4: Parallel computing and HPC

**Fri 6 Nov 2026, 14:00-15:25.** Same code, more cores.

**Before you start** (in the terminal, from the repo root):
```bash
bash scripts/update_from_template.sh
```
Then `git add mine && git commit -m "Lecture files L04" && git push`.

Everything for today is in `mine/L04/`:
- this notebook,
- `mcsim.py`, a small module with the simulation we will speed up.

There is **no in-class assignment today**. The "your turn" cells are for practice; we discuss them in class.

## By the end of today you can
- explain cores and processes, and find out how many cores you may use;
- tell embarrassingly parallel work from coupled work;
- run independent tasks in parallel with `concurrent.futures.ProcessPoolExecutor`;
- measure speedup and explain it with Amdahl's law;
- recognise when parallel code is slower (tiny tasks, more workers than cores) and fix it with `chunksize`;
- vectorise a loop with NumPy, and explain why that comes before parallelising;
- run the same tasks with Dask and read its dashboard;
- describe what an HPC cluster is and how work is submitted to it.

**Layout.** Notebook on the left. On the right, a terminal running `htop`: watch the core bars while the cells run.

## 1. Setup

Run the cell below. It imports today's module and builds the list of tasks.

In [ ]:
import inspect
import os
import time
from concurrent.futures import ProcessPoolExecutor
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mcsim

grid = mcsim.make_grid()
rows = list(grid.itertuples(index=False, name=None))   # [(task, n, rho, effect), ...]
timings = {}                                           # we store every measurement here

print("cores on this machine:   ", os.cpu_count())
print("cores we are allowed to use:", len(os.sched_getaffinity(0)))
print("number of tasks:", len(rows))
grid.head()

In the terminal, compare with `nproc` and `lscpu | head`. Then start `htop` and leave it running.

`os.cpu_count()` reports the machine. `os.sched_getaffinity(0)` reports the cores this program may use. On a shared cluster node those two numbers are usually different.

## 2. Today's problem: how wrong does the t-test get?

The two-sample t-test assumes **independent** observations. Annual rainfall, neighbouring pixels and daily streamflow are not independent: a wet year tends to follow a wet year. How wrong does the test get?

We answer this by **simulation**: make data where we know the truth, run the test many times, and count how often it rejects $H_0$ at $\alpha = 0.05$.
- With no real difference ($\delta = 0$) the rejection rate is the **false positive rate**. It should be 0.05.
- With a real difference ($\delta > 0$) the rejection rate is the **power**.

Each sample is an AR(1) series:
$$x_t = \rho\, x_{t-1} + \sqrt{1-\rho^2}\,\varepsilon_t, \qquad \varepsilon_t \sim N(0, 1)$$
$\rho$ is the lag-1 autocorrelation; $\rho = 0$ gives independent data. The second sample gets $+\delta$.

One **task** is one combination of $n$, $\rho$ and $\delta$: 5 x 5 x 4 = 100 tasks, 200 simulated data sets each. The tasks never need each other's results: this is **embarrassingly parallel** work, like processing 1000 Sentinel-2 tiles or 500 stations.

In [ ]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True, sharey=True)
for ax, rho in zip(axes, [0.0, 0.5, 0.9]):
    ax.plot(mcsim.ar1(rng, (100,), rho), lw=1)
    ax.set_ylabel(f"rho = {rho}")
axes[-1].set_xlabel("time step")
fig.suptitle("AR(1) series: higher rho, longer runs above or below the mean")
plt.show()

This is the loop we are going to speed up. Read it before running anything.

In [ ]:
print(inspect.getsource(mcsim.simulate_loop))

In [ ]:
for row in [rows[0], rows[-1]]:
    t0 = time.perf_counter()
    task, rate = mcsim.run_task(row)
    print(f"task {task:3d} (n, rho, effect = {row[1:]}): rejection rate {rate:.3f}  "
          f"in {time.perf_counter() - t0:.2f} s")

## 3. The serial baseline

First, measure. Time the first 20 tasks with an ordinary loop. Watch `htop`: how many cores are busy?

In [ ]:
t0 = time.perf_counter()
results_20 = [mcsim.run_task(r) for r in rows[:20]]
timings["serial_20"] = time.perf_counter() - t0
print(f"20 tasks: {timings['serial_20']:.1f} s")

### Your turn 3.1
Predict how long all 100 tasks will take, then run the next cell. Was your prediction too low or too high? Why?

In [ ]:
t0 = time.perf_counter()
results_serial = [mcsim.run_task(r) for r in rows]
timings["serial"] = time.perf_counter() - t0
print(f"100 tasks, serial: {timings['serial']:.1f} s")

## 4. From a loop to a pool of processes

A **process** is a running program with its own memory. A `ProcessPoolExecutor` starts a few worker processes; `map` hands each worker a task, collects what it returns, and gives the results back **in the input order**.

```python
results = [mcsim.run_task(r) for r in rows]                   # one core

with ProcessPoolExecutor(max_workers=2) as ex:               # two cores
    results = list(ex.map(mcsim.run_task, rows))
```
Run it and watch `htop`.

In [ ]:
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as ex:
    results_pool = list(ex.map(mcsim.run_task, rows))
print(f"2 workers: {time.perf_counter() - t0:.1f} s")
assert results_pool == results_serial      # same answers, task by task
print("same results as the serial loop")

Now try 1, 2, 4 and 8 workers. This cell takes about a minute.

In [ ]:
workers = [1, 2, 4, 8]
for w in workers:
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=w) as ex:
        res = list(ex.map(mcsim.run_task, rows))
    timings[w] = time.perf_counter() - t0
    assert res == results_serial
    print(f"{w} workers: {timings[w]:5.1f} s")

In [ ]:
speedup = pd.DataFrame({
    "workers": workers,
    "time (s)": [timings[w] for w in workers],
    "speedup": [timings["serial"] / timings[w] for w in workers],
})
display(speedup.round(2))

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(workers, workers, "--", color="grey", label="ideal: S = p")
ax.plot(workers, speedup["speedup"], "o-", label="measured")
ax.axvline(len(os.sched_getaffinity(0)), color="grey", lw=0.8)
ax.set_xlabel("workers p")
ax.set_ylabel("speedup S = T(serial) / T(p)")
ax.legend()
plt.show()

What to notice:
- With 2 workers on 2 cores the speedup is close to 2: the tasks are long and independent.
- 4 and 8 workers are no faster: there are still only 2 cores, so the extra workers just take turns.
- 1 worker is no faster than the plain loop: a pool only helps when there are cores for it to use.

### Amdahl's law
Real programs also have a part that cannot run in parallel: reading inputs, setting up, combining results. If a fraction $f$ of the run time can be parallelised,
$$S(p) = \frac{1}{(1-f) + f/p}, \qquad S(\infty) = \frac{1}{1-f}.$$
Today $f$ is almost 1. The cell below shows what happens when it is not.

In [ ]:
p = np.arange(1, 65)
fig, ax = plt.subplots(figsize=(6, 4))
for f in [0.5, 0.9, 0.95, 0.99]:
    ax.plot(p, 1 / ((1 - f) + f / p), label=f"f = {f}")
ax.plot(p, p, "--", color="grey", label="ideal")
ax.set_ylim(0, 40)
ax.set_xlabel("cores p")
ax.set_ylabel("speedup S(p)")
ax.legend()
plt.show()

### Your turn 4.1
A pipeline spends 10 s reading files on one core, then 90 s on work that can run in parallel. What is $f$? What speedup do you get with 8 cores, and with 64?

## 5. When parallel does not help

### Tiny tasks
Every task sent to a worker is packed up (pickled), sent, run, and the result packed and sent back. If the work itself is smaller than that trip, the pool loses.

In [ ]:
xs = range(10_000)

t0 = time.perf_counter()
r1 = [mcsim.square(x) for x in xs]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as ex:
    r2 = list(ex.map(mcsim.square, xs))
t_pool = time.perf_counter() - t0

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as ex:
    r3 = list(ex.map(mcsim.square, xs, chunksize=1000))
t_chunk = time.perf_counter() - t0

print(f"plain loop:              {t_loop:.4f} s")
print(f"pool, one item per task: {t_pool:.4f} s")
print(f"pool, chunksize=1000:    {t_chunk:.4f} s")

Rule of thumb: each task should take **at least about 0.1 s**. If your items are small, batch them with `chunksize`, so each trip carries many items.

### Vectorise before you parallelise
The loop version builds one pair of samples and calls `ttest_ind` 200 times per task. NumPy and SciPy can do all 200 at once: one array of shape (200, 2, n) and one call with `axis=1`.

In [ ]:
print(inspect.getsource(mcsim.simulate_vec))

In [ ]:
t0 = time.perf_counter()
results_vec = [mcsim.run_task(r, vectorized=True) for r in rows]
timings["vectorised"] = time.perf_counter() - t0
print(f"vectorised, 1 core: {timings['vectorised']:.2f} s "
      f"({timings['serial'] / timings['vectorised']:.0f}x faster than the serial loop)")
assert results_vec == results_serial      # same random numbers, so exactly the same answers
print("same results as the serial loop")

On one core, vectorising beats both cores by a wide margin. What does the speed buy? Precision. A rejection rate estimated from $N_{sim}$ simulations has a Monte Carlo standard error of about $\sqrt{p(1-p)/N_{sim}}$: about 0.015 for $p = 0.05$ and 200 simulations, but 0.003 with 5000. Let us rerun everything with 5000.

In [ ]:
t0 = time.perf_counter()
results_big = [mcsim.run_task(r, n_sim=5000, vectorized=True) for r in rows]
print(f"5000 simulations per task, vectorised: {time.perf_counter() - t0:.1f} s")

**Order of operations for any slow code: measure, vectorise, then parallelise.**

Vectorising has a cost: memory. The array here has $N_{sim} \times 2 \times n$ numbers of 8 bytes: 5000 x 2 x 200 x 8 bytes = 16 MB. The same trick on a stack of 10 000 x 10 000 rasters would not fit in the 8 GB of your codespace. Each worker process also has its own copy of its data: 8 workers each holding a 2 GB raster need 16 GB. Splitting data into chunks (Dask, xarray `chunks=`) is the answer to that.

### Your turn 5.1
Run the vectorised version through a 2-worker pool. Is it still worth it? Try `chunksize=10` as well.
```python
fast = partial(mcsim.run_task, vectorized=True)
with ProcessPoolExecutor(max_workers=2) as ex:
    res = list(ex.map(fast, rows))
```

In [ ]:
# your turn 5.1

## 6. Collecting results safely

Let the workers **return** their results and build one table in the main process, as we did with `ex.map`. On a cluster, where tasks run as separate jobs, give **each task its own output file** and combine them afterwards.

Do **not** let many workers write to the same file:
```python
def run_and_save(row):                     # do not do this
    task, rate = mcsim.run_task(row)
    with open("results.csv", "a") as f:    # every worker appends to one file at the same time
        f.write(f"{task},{rate}\n")
```
Lines from different workers can interleave or get lost. This is called a **race condition**: the result depends on which worker happens to write first.

## 7. Dask: the same map, with a dashboard

Dask runs tasks on a set of workers and shows you what they are doing. Start a local cluster of 2 single-threaded worker processes:

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=2, threads_per_worker=1)
client = Client(cluster)
print(client.dashboard_link)

Open the dashboard: in VS Code, go to the **Ports** tab, find port **8787**, and click the globe icon. Open **Task Stream**, then run the next cell and watch the bars appear.

In [ ]:
t0 = time.perf_counter()
futures = client.map(mcsim.run_task, rows)
results_dask = client.gather(futures)
print(f"Dask, 2 workers: {time.perf_counter() - t0:.1f} s")
assert results_dask == results_serial
print("same results as the serial loop")

In [ ]:
client.close()
cluster.close()

About as fast as the process pool, plus a little scheduling overhead. Why bother?
- The same `client.map` code runs on your laptop, a codespace or a cluster. Only the line that creates the cluster changes.
- Dask also handles data larger than memory: chunked arrays, and xarray with `chunks=`. If you use xarray with chunks, you are already using Dask.

## 8. HPC clusters in brief

A **high-performance computing (HPC) cluster** is many computers (nodes) working as one shared machine:
- you log in over SSH to a **login node**, which is only for editing files and submitting work;
- a **scheduler** (SLURM at most sites) holds a queue of jobs and starts each one on the **compute nodes** when cores and memory are free;
- every node sees the same **shared file system** (home, scratch and project space).

One node is a bigger version of your codespace: 32 to 128 cores, hundreds of GB of memory, sometimes GPUs. A cluster has hundreds of nodes.

You do not run work directly. You write a short job script that says what you need (cores, memory, time) and what to run, and submit it. Many independent tasks, like today's 100, go in as one **job array**: the scheduler runs the same script many times, each with a different task number.

Three rules: never run heavy work on the login node; ask for what you need, not the maximum; test with a few tasks before submitting thousands. Your environment travels with you: the same `pyproject.toml` and `uv.lock` rebuild it on the cluster.

## 9. What did the simulation tell us?

Back to the statistics. We use `results_big` (5000 simulations per task) from section 5.

In [ ]:
res = grid.assign(reject=[rate for _, rate in results_big])

fpr = res[res.effect == 0].pivot(index="rho", columns="n", values="reject")
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(fpr, cmap="Reds", vmin=0, vmax=0.6, origin="lower", aspect="auto")
ax.set_xticks(range(len(fpr.columns)), fpr.columns)
ax.set_yticks(range(len(fpr.index)), fpr.index)
for i in range(fpr.shape[0]):
    for j in range(fpr.shape[1]):
        ax.text(j, i, f"{fpr.iat[i, j]:.2f}", ha="center", va="center")
ax.set_xlabel("sample size n")
ax.set_ylabel("autocorrelation rho")
ax.set_title("False positive rate of the t-test (should be 0.05)")
fig.colorbar(im)
plt.show()

In [ ]:
power = res[res.effect == 0.5].pivot(index="n", columns="rho", values="reject")
ax = power.plot(marker="o", figsize=(6, 4))
ax.set_ylabel("rejection rate")
ax.set_title("Power for a true difference of 0.5 sd")
ax.legend(title="rho")
plt.show()

What to notice:
- With independent data ($\rho = 0$) the false positive rate is close to 0.05, as promised.
- With autocorrelation it is far higher, and **a larger $n$ does not fix it**.
- Why: correlated values carry less information than $n$ independent ones. The effective sample size is roughly $n_{eff} \approx n\,\frac{1-\rho}{1+\rho}$, but the test uses $n$, so its standard error is too small and its p-values too small.
- The "power" at high $\rho$ and small $n$ looks good only because the test rejects too often anyway.

Where this matters: trends in annual rainfall or streamflow, comparing pixels from the same scene, gauges in the same basin. Remedies exist (block bootstrap, adjusting for $n_{eff}$, prewhitening, mixed models); the first step is knowing the problem is there.

### Your turn 9.1
Predict the false positive rate for $\rho = 0.95$, then check it with the vectorised code for every $n$, with $\delta = 0$ and 5000 simulations.

In [ ]:
# your turn 9.1

## 10. Summary

| Tool or idea | Use it when | Today |
|---|---|---|
| measure first | always | `time.perf_counter()` around the loop |
| `ProcessPoolExecutor` | many independent tasks, each taking 0.1 s or more | `ex.map(mcsim.run_task, rows)`: about 2x on 2 cores |
| no more workers than cores | always | 4 and 8 workers were no faster than 2 |
| `chunksize` | many small items | `ex.map(square, xs, chunksize=1000)` |
| NumPy vectorisation | the same operation on many values | `ttest_ind(a, b, axis=1)`: by far the biggest gain |
| one output per task | collecting results from many workers or jobs | return values, or one file per task |
| Dask `client.map` | the same code on a laptop, a codespace or a cluster; data bigger than memory | dashboard on port 8787 |
| HPC cluster | more cores than one machine has | job scripts, a scheduler, job arrays |

Commit and push this notebook before you close the codespace: `git add mine/L04 && git commit -m "L04" && git push`.

**Reading:** the Python [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html) documentation, and Dask's [best practices](https://docs.dask.org/en/stable/best-practices.html).